# GAVE2 Task 1 training on Kaggle

Before running: Notebook settings (right panel) -> Accelerator: **GPU T4 x2**, Internet: **On**.
Also make sure the `GH_TOKEN` secret is attached to this notebook (Add-ons -> Secrets), and the
`gave2-preliminary` dataset is added as an input (Add Input -> Your Datasets).

## Cell 1: clone the repo

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GH_TOKEN")
repo_url = f"https://{token}@github.com/aaronjji/gave2-challenge.git"

if not os.path.exists("/kaggle/working/gave2-challenge"):
    os.system(f"git clone --recurse-submodules {repo_url} /kaggle/working/gave2-challenge")
else:
    os.system("cd /kaggle/working/gave2-challenge && git pull && git submodule update --init --recursive")

os.chdir("/kaggle/working/gave2-challenge")
print(os.getcwd())

## Cell 2: install dependencies
(torch/torchvision are preinstalled on Kaggle -- skip those.)

In [ ]:
os.system("pip install -q albumentations opencv-python-headless scikit-image scipy networkx sknw huggingface_hub safetensors")

## Cell 3: point at the uploaded dataset
Lists the input dir first so you can see the real structure if the assert below fails --
Kaggle's zip auto-extract occasionally flattens or nests differently than expected.

In [ ]:
candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-preliminary/GAVE2_preliminary",
    "/kaggle/input/gave2-preliminary/GAVE2_preliminary",
    "/kaggle/input/gave2-preliminary",
]
os.system("find /kaggle/input -maxdepth 4")

DATA_ROOT = None
for c in candidates:
    if os.path.exists(f"{c}/training/images"):
        DATA_ROOT = c
        break
assert DATA_ROOT is not None, f"Dataset not found in any of {candidates} -- check the find output above and set DATA_ROOT manually"
print("DATA_ROOT =", DATA_ROOT)

## Cell 4: (optional) resume from a previous session's checkpoint
If you added a previous notebook version's output as an input, copy its checkpoint into place
before training so `--resume` picks it up.

In [ ]:
import shutil
from pathlib import Path

PREV_CHECKPOINT = None  # e.g. "/kaggle/input/your-notebook-output-vN/runs/task1/fold0/latest.pth"
if PREV_CHECKPOINT and os.path.exists(PREV_CHECKPOINT):
    dst = Path("runs/task1/fold0/latest.pth")
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(PREV_CHECKPOINT, dst)
    print(f"Restored checkpoint from {PREV_CHECKPOINT}")

## Cell 5: train one fold
Kaggle sessions cap out around 9-12h; `--max-seconds` leaves a safety margin so the run
checkpoints and exits cleanly rather than getting killed mid-step. Re-run this same cell
(with `--resume`) if you need more epochs than one session allows -- it picks up from
`runs/task1/fold{N}/latest.pth`.

In [ ]:
FOLD = 0
os.system(
    f"python -u src/train_task1.py "
    f"--fold {FOLD} --data-root {DATA_ROOT} "
    f"--epochs 60 --steps-per-epoch 50 --num-workers 2 "
    f"--checkpoint-every-epochs 2 --max-seconds 30000 "
    f"--out-dir runs/task1 --resume"
)

## Cell 6: repeat for the other folds
Change `FOLD` to 1, 2, 3, 4 and re-run Cell 5 (each fold trains independently). Once
satisfied, **Save Version** to persist `runs/` as this notebook's output, then download the
checkpoints (or add this version as an input to a follow-up inference notebook/session).

## Cell 7: Task 2 -- upload registered FFA data first
Task 2 needs CFP<->FFA registration (they are NOT pre-aligned -- verified empirically). Registration was already run locally (MINIMA sp_lg, 100/100 pairs succeeded on both splits, zero fallbacks). Upload the local `data/registered/` folder (~232MB) as a **new private Kaggle Dataset** the same way you did the main one, add it as an input to this notebook, then run the cell below.

In [ ]:
reg_candidates = [
    "/kaggle/input/datasets/aaronajit/gave2-registered/registered",
    "/kaggle/input/gave2-registered/registered",
    "/kaggle/input/gave2-registered",
]
os.system("find /kaggle/input -maxdepth 5")

FFA_ROOT = None
for c in reg_candidates:
    if os.path.exists(f"{c}/training/FFA_A"):
        FFA_ROOT = c
        break
assert FFA_ROOT is not None, f"Registered FFA data not found in any of {reg_candidates} -- check the find output above"
print("FFA_ROOT =", FFA_ROOT)

## Cell 8: train Task 2 (fold 0, warm-started from Task 1)
Warm-starts CMRRWNet's RGB encoder + decoder + second_u from the Task 1 checkpoint trained in Cell 5 (180/180 tensors transfer cleanly -- verified locally). Same session-budget / resume pattern as Task 1.

In [ ]:
FOLD2 = 0
os.system(
    f"python -u src/train_task2.py "
    f"--fold {FOLD2} --data-root {DATA_ROOT} --ffa-root {FFA_ROOT} "
    f"--warm-start-task1 runs/task1/fold0/latest.pth "
    f"--epochs 60 --steps-per-epoch 50 --num-workers 2 "
    f"--checkpoint-every-epochs 2 --max-seconds 30000 "
    f"--out-dir runs/task2 --resume"
)